In [ ]:
import os
import torch
import pandas as pd

from datasets import Dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import evaluate


# ============================================================
# Configuration
# ============================================================

MODEL_NAME = "openai/whisper-small"
LANGUAGE = "english"
TASK = "transcribe"

AUDIO_DIR = "./data/audio"
CSV_FILE = "./data/metadata.csv"

OUTPUT_DIR = "./whisper-finetuned"

# For a small demo dataset, keep this relatively small.
NUM_TRAIN_EPOCHS = 3

# Adjust depending on GPU memory.
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4

LEARNING_RATE = 1e-5


# ============================================================
# Load dataset
# ============================================================

df = pd.read_csv(CSV_FILE)

# Convert relative filenames into full paths.
df["audio"] = df["audio"].apply(
    lambda x: os.path.join(AUDIO_DIR, x)
)

# Rename transcript column if necessary.
df = df[["audio", "text"]]

# Basic validation.
for path in df["audio"]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Audio file not found: {path}")

if len(df) < 10:
    print(
        "WARNING: You have fewer than 10 examples. "
        "This is fine for a demo, but is usually far too "
        "small for useful fine-tuning."
    )

dataset = Dataset.from_pandas(df)

# Whisper expects 16 kHz audio.
dataset = dataset.cast_column(
    "audio",
    Audio(sampling_rate=16_000)
)

# Create train/evaluation split.
split = dataset.train_test_split(
    test_size=0.1,
    seed=42
)

train_dataset = split["train"]
eval_dataset = split["test"]

print("Training examples:", len(train_dataset))
print("Evaluation examples:", len(eval_dataset))


# ============================================================
# Processor
# ============================================================

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    language=LANGUAGE,
    task=TASK,
)


# ============================================================
# Prepare audio + transcripts
# ============================================================

def prepare_example(example):
    audio = example["audio"]

    # Convert audio waveform into Whisper input features.
    inputs = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
    )

    # Convert transcript into token IDs.
    labels = processor.tokenizer(
        example["text"]
    ).input_ids

    return {
        "input_features": inputs.input_features[0],
        "labels": labels,
    }


train_dataset = train_dataset.map(
    prepare_example,
    remove_columns=train_dataset.column_names,
)

eval_dataset = eval_dataset.map(
    prepare_example,
    remove_columns=eval_dataset.column_names,
)


# ============================================================
# Data collator
# ============================================================

class DataCollatorSpeechSeq2Seq:
    def __init__(self, processor):
        self.processor = processor
        self.feature_extractor = processor.feature_extractor
        self.tokenizer = processor.tokenizer

    def __call__(self, features):
        # Pad audio features.
        input_features = [
            {"input_features": f["input_features"]}
            for f in features
        ]

        batch = self.feature_extractor.pad(
            input_features,
            return_tensors="pt",
        )

        # Pad transcript labels.
        label_features = [
            {"input_ids": f["labels"]}
            for f in features
        ]

        labels_batch = self.tokenizer.pad(
            label_features,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"]

        # Replace padding with -100 so that the loss ignores it.
        labels = labels.masked_fill(
            labels_batch.attention_mask.ne(1),
            -100,
        )

        # Remove BOS if already included.
        if (
            labels.shape[1] > 0
            and (labels[:, 0] == self.tokenizer.bos_token_id).all()
        ):
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch


data_collator = DataCollatorSpeechSeq2Seq(processor)


# ============================================================
# Load Whisper
# ============================================================

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME
)

# Tell Whisper explicitly what language/task we're training for.
model.generation_config.language = LANGUAGE
model.generation_config.task = TASK
model.generation_config.forced_decoder_ids = None

# Useful when using gradient checkpointing.
model.config.use_cache = False


# ============================================================
# Evaluation metric
# ============================================================

wer_metric = evaluate.load("wer")


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 with padding token.
    label_ids[label_ids == -100] = (
        processor.tokenizer.pad_token_id
    )

    pred_str = processor.batch_decode(
        pred_ids,
        skip_special_tokens=True,
    )

    label_str = processor.batch_decode(
        label_ids,
        skip_special_tokens=True,
    )

    wer = 100 * wer_metric.compute(
        predictions=pred_str,
        references=label_str,
    )

    return {
        "wer": wer
    }


# ============================================================
# Training configuration
# ============================================================

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_TRAIN_EPOCHS,

    # Evaluation
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="steps",
    eval_steps=100,

    # Generation
    predict_with_generate=True,
    generation_max_length=225,

    # Checkpoints
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    # Select best checkpoint by WER.
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    # Performance
    gradient_checkpointing=True,

    # Use FP16 on NVIDIA GPU.
    fp16=torch.cuda.is_available(),

    # Logging
    logging_steps=10,
    report_to=["tensorboard"],

    # Don't upload anything to Hugging Face.
    push_to_hub=False,

    # Required by some Transformers versions when
    # using generation during evaluation.
    remove_unused_columns=False,
)


# ============================================================
# Trainer
# ============================================================

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)


# ============================================================
# Train
# ============================================================

print("Starting fine-tuning...")

trainer.train()


# ============================================================
# Save model
# ============================================================

trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print(f"Model saved to: {OUTPUT_DIR}")


In [ ]:
import torch
import librosa

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
)


MODEL_DIR = "./whisper-finetuned"
AUDIO_FILE = "./data/audio/test.wav"

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = WhisperProcessor.from_pretrained(MODEL_DIR)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_DIR
).to(device)

model.eval()

# Load audio at Whisper's expected sampling rate.
audio, sr = librosa.load(
    AUDIO_FILE,
    sr=16_000,
)

inputs = processor(
    audio,
    sampling_rate=16_000,
    return_tensors="pt",
)

input_features = inputs.input_features.to(device)

with torch.no_grad():
    predicted_ids = model.generate(
        input_features,
        language="english",
        task="transcribe",
    )

transcription = processor.batch_decode(
    predicted_ids,
    skip_special_tokens=True,
)[0]

print("Transcription:")
print(transcription)
